In [1]:
!pip install -q sagemaker s3fs pandas boto3 joblib

In [2]:
import sagemaker
from sagemaker.sklearn import SKLearn
import boto3
import os
import time

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
default_bucket = sess.default_bucket()
region = sess.boto_region_name

# Caminhos S3
data_bucket = 'ons-risk-prediction-data-674650987717'
data_prefix = 'export'
s3_input_data_path = f's3://{data_bucket}/{data_prefix}/'
s3_model_output_path = f's3://{default_bucket}/ons-risk-prediction/brf-models/'

print(f"✓ SageMaker configurado")
print(f"  Role: {role}")
print(f"  Bucket: {default_bucket}")
print(f"  Dados: {s3_input_data_path}")
print(f"  Saída: {s3_model_output_path}")

✓ Diretório 'source_brf_fixed' criado
✓ requirements.txt criado com versões fixadas

Versões:
  - scikit-learn: 1.2.2
  - imbalanced-learn: 0.10.1 (compatível)


In [3]:
%%writefile {source_dir}/train_balanced_rf.py
import argparse
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from imblearn.ensemble import BalancedRandomForestClassifier
import joblib
import sys
import traceback
import time

if __name__ == '__main__':
    parser = argparse.ArgumentParser()
    parser.add_argument('--model-dir', type=str, default=os.environ.get('SM_MODEL_DIR'))
    parser.add_argument('--train', type=str, default=os.environ.get('SM_CHANNEL_TRAIN'))
    
    # Hiperparâmetros
    parser.add_argument('--n_estimators', type=int, default=300)
    parser.add_argument('--max_depth', type=int, default=15)
    parser.add_argument('--min_samples_leaf', type=int, default=2)
    parser.add_argument('--max_features', type=str, default='sqrt')
    parser.add_argument('--criterion', type=str, default='gini')
    parser.add_argument('--sampling_strategy', type=str, default='not majority')
    parser.add_argument('--replacement', type=str, default='True')
    parser.add_argument('--bootstrap', type=str, default='False')
    parser.add_argument('--target_column_index', type=int, default=2)
    
    args = parser.parse_args()
    args.replacement = args.replacement.lower() == 'true'
    args.bootstrap = args.bootstrap.lower() == 'true'
    
    print("=" * 80)
    print("BALANCED RANDOM FOREST - ONS RISK PREDICTION")
    print("=" * 80)
    print(f"Target: _COL_{args.target_column_index} (baixo/medio/alto)")
    print(f"n_estimators: {args.n_estimators}")
    print(f"max_depth: {args.max_depth}")
    print(f"min_samples_leaf: {args.min_samples_leaf}")
    print(f"max_features: {args.max_features}")
    print(f"criterion: {args.criterion}")
    print(f"sampling_strategy: {args.sampling_strategy}")
    print(f"replacement: {args.replacement}")
    print(f"bootstrap: {args.bootstrap}")
    print("=" * 80)
    
    try:
        start_time = time.time()
        
        # ==================== CARREGAR DADOS ====================
        print("\n[1/6] CARREGANDO DADOS...")
        input_files = [os.path.join(args.train, f) 
                      for f in os.listdir(args.train) 
                      if f.endswith('.parquet')]
        
        if not input_files:
            raise ValueError(f"Nenhum arquivo parquet em {args.train}")
        
        print(f"Arquivos encontrados: {len(input_files)}")
        
        dfs = []
        for i, f in enumerate(input_files, 1):
            df_temp = pd.read_parquet(f, engine='pyarrow')
            print(f"  ✓ [{i}/{len(input_files)}] {os.path.basename(f)}: {df_temp.shape}")
            dfs.append(df_temp)
        
        df = pd.concat(dfs, ignore_index=True)
        print(f"\n✓ Dataset completo: {df.shape}")
        print(f"  Tempo de leitura: {time.time() - start_time:.1f}s")
        
        del dfs
        
        # ==================== SEPARAR TARGET E FEATURES ====================
        print("\n[2/6] SEPARANDO TARGET E FEATURES...")
        
        target_col = df.columns[args.target_column_index]
        print(f"  Coluna target: {target_col} (índice {args.target_column_index})")
        
        # Features: todas colunas numéricas exceto target
        feature_cols = [col for col in df.columns if col != target_col]
        X = df[feature_cols].select_dtypes(include=[np.number])
        y = df[target_col]
        
        print(f"  Features: {X.shape[1]} colunas numéricas")
        print(f"  Amostras: {len(y):,}")
        
        del df
        
        # ==================== PRÉ-PROCESSAR TARGET ====================
        print("\n[3/6] PRÉ-PROCESSANDO TARGET...")
        
        # Remover NaNs do target
        if y.isnull().any():
            n_nans = y.isnull().sum()
            print(f"  Removendo {n_nans:,} NaNs do target")
            valid_idx = y.dropna().index
            X = X.loc[valid_idx]
            y = y.loc[valid_idx]
        
        print(f"  Tipo do target: {y.dtype}")
        print(f"  Valores únicos: {sorted(y.unique())}")
        
        # Mapear para inteiros
        if y.dtype == 'object' or y.dtype.name == 'category':
            mapeamento = {'baixo': 0, 'medio': 1, 'alto': 2}
            y_encoded = y.map(mapeamento)
            
            if y_encoded.isnull().any():
                unmapped = y[y_encoded.isnull()].unique()
                raise ValueError(f"❌ Valores não mapeados no target: {unmapped}")
        else:
            # Assumir que já está codificado
            y_encoded = y.astype(int)
        
        # Validar 3 classes
        unique_classes = sorted(y_encoded.unique())
        if len(unique_classes) != 3:
            raise ValueError(f"❌ Esperadas 3 classes, encontradas {len(unique_classes)}: {unique_classes}")
        
        if unique_classes != [0, 1, 2]:
            print(f"  ⚠️ Classes não são [0,1,2]: {unique_classes}")
            print(f"  Remapeando para [0,1,2]...")
            remap = {old: new for new, old in enumerate(unique_classes)}
            y_encoded = y_encoded.map(remap)
        
        # Distribuição
        print(f"\n  ✅ Distribuição do target:")
        dist = y_encoded.value_counts().sort_index()
        
        for idx, count in dist.items():
            classe = ['baixo', 'medio', 'alto'][idx]
            pct = count / len(y_encoded) * 100
            print(f"    {idx} ({classe:6s}): {count:5,} amostras ({pct:5.1f}%)")
        
        # ==================== PRÉ-PROCESSAR FEATURES ====================
        print("\n[4/6] PRÉ-PROCESSANDO FEATURES...")
        print(f"  Shape inicial: {X.shape}")
        
        # 1. Remover colunas vazias (100% NaN)
        X = X.dropna(axis=1, how='all')
        print(f"  Após remover colunas vazias: {X.shape}")
        
        # 2. Remover colunas com variância zero
        variance = X.var()
        cols_to_keep = variance[variance > 0].index
        removed_var_zero = len(X.columns) - len(cols_to_keep)
        X = X[cols_to_keep]
        print(f"  Após remover {removed_var_zero} colunas com var=0: {X.shape}")
        
        # 3. Preencher NaNs restantes
        if X.isnull().sum().any():
            n_nans = X.isnull().sum().sum()
            print(f"  Preenchendo {n_nans:,} NaNs com mediana")
            X = X.fillna(X.median())
            X = X.fillna(0)  # Caso mediana seja NaN
        
        # 4. Tratar infinitos
        X = X.replace([np.inf, -np.inf], np.nan)
        if X.isnull().sum().any():
            n_inf = X.isnull().sum().sum()
            print(f"  Substituindo {n_inf:,} valores infinitos")
            X = X.fillna(X.median()).fillna(0)
        
        print(f"  ✓ Shape final: {X.shape}")
        print(f"    Verificação: NaNs={X.isnull().sum().sum()}, Infs={np.isinf(X.values).sum()}")
        
        # 5. Escalar features
        print("\n  Escalando features com StandardScaler...")
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(X)
        
        # Limpeza final pós-scaling
        X_scaled = np.nan_to_num(X_scaled, nan=0.0, posinf=0.0, neginf=0.0)
        print(f"  ✓ Features escaladas: shape={X_scaled.shape}")
        
        # ==================== TREINAR MODELO ====================
        print("\n[5/6] TREINANDO BALANCED RANDOM FOREST...")
        print(f"  Configuração do modelo:")
        print(f"    n_estimators: {args.n_estimators}")
        print(f"    max_depth: {args.max_depth}")
        print(f"    min_samples_leaf: {args.min_samples_leaf}")
        print(f"    max_features: {args.max_features}")
        print(f"    criterion: {args.criterion}")
        print(f"    sampling_strategy: {args.sampling_strategy}")
        print(f"    replacement: {args.replacement}")
        print(f"    bootstrap: {args.bootstrap}")
        
        train_start = time.time()
        
        model = BalancedRandomForestClassifier(
            n_estimators=args.n_estimators,
            max_depth=args.max_depth if args.max_depth > 0 else None,
            min_samples_leaf=args.min_samples_leaf,
            max_features=args.max_features,
            criterion=args.criterion,
            sampling_strategy=args.sampling_strategy,
            replacement=args.replacement,
            bootstrap=args.bootstrap,
            random_state=42,
            n_jobs=-1,
            verbose=2
        )
        
        print(f"\n  Iniciando fit() com {len(y_encoded):,} amostras e {X_scaled.shape[1]} features...")
        model.fit(X_scaled, y_encoded.values)
        
        train_time = time.time() - train_start
        print(f"\n  ✓ Treinamento concluído!")
        print(f"    Tempo: {train_time:.1f}s ({train_time/60:.1f} min)")
        
        # ==================== SALVAR ARTEFATOS ====================
        print("\n[6/6] SALVANDO ARTEFATOS...")
        
        model_path = os.path.join(args.model_dir, 'brf-model.joblib')
        scaler_path = os.path.join(args.model_dir, 'scaler_brf.joblib')
        features_path = os.path.join(args.model_dir, 'feature_names.joblib')
        mapping_path = os.path.join(args.model_dir, 'target_mapping.joblib')
        
        print(f"  Salvando modelo: {model_path}")
        joblib.dump(model, model_path)
        
        print(f"  Salvando scaler: {scaler_path}")
        joblib.dump(scaler, scaler_path)
        
        print(f"  Salvando feature names: {features_path}")
        joblib.dump(X.columns.tolist(), features_path)
        
        print(f"  Salvando target mapping: {mapping_path}")
        mapeamento = {'baixo': 0, 'medio': 1, 'alto': 2}
        joblib.dump(mapeamento, mapping_path)
        
        print("  ✓ Todos os artefatos salvos com sucesso!")
        
        # ==================== IMPORTÂNCIA DAS FEATURES ====================
        if hasattr(model, 'feature_importances_'):
            print("\n  📊 TOP 15 FEATURES MAIS IMPORTANTES:")
            importances = model.feature_importances_
            indices = np.argsort(importances)[::-1]
            
            for i in range(min(15, len(importances))):
                idx = indices[i]
                print(f"    {i+1:2d}. {X.columns[idx]:15s} {importances[idx]:.6f}")
        
        # ==================== RESUMO FINAL ====================
        total_time = time.time() - start_time
        
        print("\n" + "=" * 80)
        print("✅ TREINAMENTO CONCLUÍDO COM SUCESSO!")
        print("=" * 80)
        print(f"⏱️  TEMPO TOTAL: {total_time:.1f}s ({total_time/60:.1f} minutos)")
        print(f"   - Leitura de dados: {train_start - start_time:.1f}s")
        print(f"   - Pré-processamento: {train_start - start_time:.1f}s")
        print(f"   - Treinamento: {train_time:.1f}s")
        print()
        print(f"📊 DATASET:")
        print(f"   - Total de amostras: {len(y_encoded):,}")
        print(f"   - Features utilizadas: {X.shape[1]}")
        print(f"   - Features removidas: {removed_var_zero + 7} (var=0 + NaN)")
        print()
        print(f"🎯 DISTRIBUIÇÃO DAS CLASSES:")
        for idx, count in dist.items():
            classe = ['Baixo', 'Médio', 'Alto'][idx]
            pct = count / len(y_encoded) * 100
            print(f"   - {classe:6s}: {count:5,} ({pct:5.1f}%)")
        print()
        print(f"💾 ARTEFATOS SALVOS:")
        print(f"   - brf-model.joblib (modelo treinado)")
        print(f"   - scaler_brf.joblib (scaler para features)")
        print(f"   - feature_names.joblib (nomes das {X.shape[1]} features)")
        print(f"   - target_mapping.joblib (mapeamento baixo/medio/alto)")
        print("=" * 80)
        
        # Retornar sucesso
        sys.exit(0)
        
    except Exception as e:
        print("\n" + "=" * 80)
        print("❌ ERRO DURANTE O TREINAMENTO")
        print("=" * 80)
        print(f"\nTipo: {type(e).__name__}")
        print(f"Mensagem: {str(e)}\n")
        traceback.print_exc()
        sys.exit(255)

Overwriting source_brf_fixed/train_balanced_rf.py


In [4]:
import sagemaker
from sagemaker.sklearn import SKLearn
import boto3
import time

sess = sagemaker.Session()
role = sagemaker.get_execution_role()
default_bucket = sess.default_bucket()
region = sess.boto_region_name

# Caminhos S3
data_bucket = 'ons-risk-prediction-data-674650987717'
data_prefix = 'export'
s3_input_data_path = f's3://{data_bucket}/{data_prefix}/'
s3_model_output_path = f's3://{default_bucket}/ons-risk-prediction/brf-models/'

print("=" * 80)
print("CONFIGURAÇÃO DO SAGEMAKER")
print("=" * 80)
print(f"✓ Role: {role}")
print(f"✓ Bucket: {default_bucket}")
print(f"✓ Região: {region}")
print(f"✓ Dados de entrada: {s3_input_data_path}")
print(f"✓ Saída do modelo: {s3_model_output_path}")
print("=" * 80)

/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml
CONFIGURAÇÃO DO SAGEMAKER
✓ Role: arn:aws:iam::674650987717:role/service-role/SageMaker-DataEngineer
✓ Bucket: sagemaker-us-east-1-674650987717
✓ Região: us-east-1
✓ Dados de entrada: s3://ons-risk-prediction-data-674650987717/export/
✓ Saída do modelo: s3://sagemaker-us-east-1-674650987717/ons-risk-prediction/brf-models/


In [5]:
hyperparameters = {
    'n_estimators': 300,
    'max_depth': 15,
    'min_samples_leaf': 2,
    'max_features': 'sqrt',
    'criterion': 'gini',
    'sampling_strategy': 'not majority',
    'replacement': 'True',
    'bootstrap': 'False',  # NOVO: evitar warning
    'target_column_index': 2  # _COL_2 (baixo/medio/alto)
}

print("\n📋 HIPERPARÂMETROS:")
print("=" * 80)
for key, value in hyperparameters.items():
    print(f"  {key:25s}: {value}")
print("=" * 80)


📋 HIPERPARÂMETROS:
  n_estimators             : 300
  max_depth                : 15
  min_samples_leaf         : 2
  max_features             : sqrt
  criterion                : gini
  sampling_strategy        : not majority
  replacement              : True
  bootstrap                : False
  target_column_index      : 2


In [6]:
estimator = SKLearn(
    entry_point='train_balanced_rf.py',
    source_dir=source_dir,
    role=role,
    instance_count=1,
    instance_type='ml.m5.2xlarge',
    framework_version='1.2-1',
    py_version='py3',
    hyperparameters=hyperparameters,
    output_path=s3_model_output_path,
    sagemaker_session=sess,
    max_run=3600,  # 1 hora max
    use_spot_instances=True,
    max_wait=3600
)

print("✓ Estimador SKLearn criado")
print(f"  Instância: ml.m5.2xlarge (8 vCPUs, 32GB RAM)")
print(f"  Framework: sklearn 1.2-1")
print(f"  Spot instances: Ativado (economia de até 70%)")

✓ Estimador SKLearn criado
  Instância: ml.m5.2xlarge (8 vCPUs, 32GB RAM)
  Framework: sklearn 1.2-1
  Spot instances: Ativado (economia de até 70%)


In [7]:
train_input = sagemaker.inputs.TrainingInput(
    s3_data=s3_input_data_path,
    distribution='FullyReplicated',
    content_type='parquet',
    s3_data_type='S3Prefix'
)

print("✓ Input de dados configurado")

✓ Input de dados configurado


In [8]:
timestamp = time.strftime("%Y%m%d-%H%M%S", time.gmtime())
job_name = f'brf-ons-risk-fixed-{timestamp}'

print("\n" + "=" * 80)
print("🚀 INICIANDO TREINAMENTO")
print("=" * 80)
print(f"Job name: {job_name}")
print(f"Target: _COL_2 (baixo/medio/alto)")
print(f"Dataset: ~5,794 amostras")
print(f"Features: ~20 colunas numéricas")
print("=" * 80)
print("\n⏳ Aguarde... Treinamento estimado: 5-10 minutos\n")

# INICIAR TREINAMENTO
estimator.fit({'train': train_input}, job_name=job_name, wait=True)

INFO:sagemaker:Creating training-job with name: brf-ons-risk-fixed-20251020-212249



🚀 INICIANDO TREINAMENTO
Job name: brf-ons-risk-fixed-20251020-212249
Target: _COL_2 (baixo/medio/alto)
Dataset: ~5,794 amostras
Features: ~20 colunas numéricas

⏳ Aguarde... Treinamento estimado: 5-10 minutos

2025-10-20 21:22:49 Starting - Starting the training job......
2025-10-20 21:23:52 Downloading - Downloading input data...
2025-10-20 21:24:07 Downloading - Downloading the training image...
2025-10-20 21:24:53 Training - Training image download completed. Training in progress..../miniconda3/lib/python3.9/site-packages/sagemaker_containers/_server.py:22: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
2025-10-20 21:25:08,584 sagemaker-containers INFO     Imported framework sagemaker_sklearn_container.training
2025-10-20 21:25:08,588 sagemaker-training-toolkit INFO  

In [9]:
model_uri = estimator.model_data

print("\n" + "=" * 80)
print("✅ TREINAMENTO CONCLUÍDO COM SUCESSO!")
print("=" * 80)
print(f"\n📦 Modelo treinado:")
print(f"   {model_uri}")
print(f"\n💾 Para baixar o modelo:")
print(f"   !aws s3 cp {model_uri} ./modelo_brf.tar.gz")
print(f"   !tar -xzf modelo_brf.tar.gz")
print(f"\n📁 Artefatos que serão extraídos:")
print(f"   - brf-model.joblib        (modelo treinado)")
print(f"   - scaler_brf.joblib       (StandardScaler)")
print(f"   - feature_names.joblib    (lista de features)")
print(f"   - target_mapping.joblib   (baixo:0, medio:1, alto:2)")
print("=" * 80)


✅ TREINAMENTO CONCLUÍDO COM SUCESSO!

📦 Modelo treinado:
   s3://sagemaker-us-east-1-674650987717/ons-risk-prediction/brf-models/brf-ons-risk-fixed-20251020-212249/output/model.tar.gz

💾 Para baixar o modelo:
   !aws s3 cp s3://sagemaker-us-east-1-674650987717/ons-risk-prediction/brf-models/brf-ons-risk-fixed-20251020-212249/output/model.tar.gz ./modelo_brf.tar.gz
   !tar -xzf modelo_brf.tar.gz

📁 Artefatos que serão extraídos:
   - brf-model.joblib        (modelo treinado)
   - scaler_brf.joblib       (StandardScaler)
   - feature_names.joblib    (lista de features)
   - target_mapping.joblib   (baixo:0, medio:1, alto:2)


In [10]:
print("\n🔽 Baixando modelo...")
!aws s3 cp {model_uri} ./modelo_brf.tar.gz
!tar -xzf modelo_brf.tar.gz

print("\n📦 Carregando artefatos...")
import joblib

model = joblib.load('brf-model.joblib')
scaler = joblib.load('scaler_brf.joblib')
feature_names = joblib.load('feature_names.joblib')
target_mapping = joblib.load('target_mapping.joblib')

print(f"✓ Modelo carregado: {type(model).__name__}")
print(f"✓ Features: {len(feature_names)}")
print(f"✓ Mapeamento target: {target_mapping}")
print(f"✓ N estimators: {model.n_estimators}")
print(f"✓ Max depth: {model.max_depth}")

print("\n🎯 MODELO PRONTO PARA USO!")


🔽 Baixando modelo...
download: s3://sagemaker-us-east-1-674650987717/ons-risk-prediction/brf-models/brf-ons-risk-fixed-20251020-212249/output/model.tar.gz to ./modelo_brf.tar.gz
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'
tar: Ignoring unknown extended header keyword `LIBARCHIVE.creationtime'

📦 Carregando artefatos...


/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:8                                                                                    │
│                                                                                                  │
│    5 print("\n📦 Carregando artefatos...")                                                       │
│    6 import joblib                                                                               │
│    7                                                                                             │
│ ❱  8 model = joblib.load('brf-model.joblib')                                                     │
│    9 scaler = joblib.load('scaler_brf.joblib')                                                   │
│   10 feature_names = joblib.load('feature_names.joblib')                                         │
│   11 target_mapping = joblib.load('target_mapping.joblib')                                       │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/joblib/numpy_pickle.py:749 in │
│ load                                                                                             │
│                                                                                                  │
│   746 │   │   │   │   # A memory-mapped array has to be mapped with the endianness               │
│   747 │   │   │   │   # it has been written with. Other arrays are coerced to the                │
│   748 │   │   │   │   # native endianness of the host system.                                    │
│ ❱ 749 │   │   │   │   obj = _unpickle(                                                           │
│   750 │   │   │   │   │   fobj,                                                                  │
│   751 │   │   │   │   │   ensure_native_byte_order=ensure_native_byte_order,                     │
│   752 │   │   │   │   │   filename=filename,                                                     │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/joblib/numpy_pickle.py:626 in │
│ _unpickle                                                                                        │
│                                                                                                  │
│   623 │   )                                                                                      │
│   624 │   obj = None                                                                             │
│   625 │   try:                                                                                   │
│ ❱ 626 │   │   obj = unpickler.load()                                                             │
│   627 │   │   if unpickler.compat_mode:                                                          │
│   628 │   │   │   warnings.warn(                                                                 │
│   629 │   │   │   │   "The file '%s' has been generated with a "                                 │
│                                                                                                  │
│ /home/ec2-user/anaconda3/envs/python3/lib/python3.10/pickle.py:1213 in load                      │
│                                                                                                  │
│   1210 │   │   │   │   if not key:                                                               │
│   1211 │   │   │   │   │   raise EOFError                                                        │
│   1212 │   │   │   │   assert isinstance(key, bytes_types)                                       │
│ ❱ 1213 │   │   │   │   dispatch[key[0]](self)                                                    │
│   1214 │   │   except _Stop as stopinst:                    